# Medical MTEB Evaluation

This notebook is self-contained and evaluates a reduced set of models on the medical MTEB tasks.

**Selected models**
- CQG-MBQA
- QAEmb-MBQA
- QIME
- LDIR-UAE-500
- bag_of_words
- google-bert/bert-base-uncased

## Setup

In [ ]:
from pathlib import Path
import os
import sys
import json
import warnings

import mteb
import numpy as np
import pandas as pd
import torch
from sentence_transformers import SentenceTransformer

warnings.filterwarnings("ignore")

cwd = Path.cwd()
if (cwd / "imec_model.py").exists() and (cwd / "anchor_sim_model.py").exists():
    framework_dir = cwd
elif (cwd / "framework" / "imec_model.py").exists():
    framework_dir = cwd / "framework"
elif (cwd.parent / "framework" / "imec_model.py").exists():
    framework_dir = cwd.parent / "framework"
else:
    raise FileNotFoundError("Could not locate the framework directory.")

repo_root = framework_dir.parent
checkpoints_dir = repo_root / "checkpoints"
results_dir = repo_root / "results"
model_outputs_dir = results_dir / "evaluation" / "model_outputs"
report_dir = results_dir / "medical_mteb_retrieval_metrics"

if str(framework_dir) not in sys.path:
    sys.path.insert(0, str(framework_dir))

from imec_model import IMECClassifier, IMECMTEBModelWrapper
from anchor_sim_model import cos_simi_model
from utils import BagOfTokenEncoder

device = "cuda" if torch.cuda.is_available() else "cpu"

print("Framework directory:", framework_dir)
print("Device:", device)

## Evaluation Logic

In [ ]:
class SBERTEncodingModelGeneral:
    def __init__(self, model_name, device=device):
        self.model = SentenceTransformer(model_name, device=device)

    def encode(self, sentences, **kwargs):
        unsupported_keys = [
            "task_name", "task_category", "data_split", "max_seq_length",
            "return_numpy", "batch_size", "normalize_embeddings", "prompt_type"
        ]
        for key in unsupported_keys:
            kwargs.pop(key, None)

        kwargs.setdefault("show_progress_bar", False)
        convert_to_tensor = kwargs.pop("convert_to_tensor", True)
        convert_to_numpy = kwargs.pop("convert_to_numpy", False)

        embeddings = self.model.encode(sentences, convert_to_tensor=True, **kwargs)

        if hasattr(embeddings, "cpu"):
            embeddings = embeddings.cpu()

        if convert_to_numpy or not convert_to_tensor:
            return embeddings.numpy() if hasattr(embeddings, "numpy") else embeddings
        return embeddings


class LDIRModelWrapper:
    def __init__(self, anchor_file_path, backbone_model="WhereIsAI/UAE-Large-V1", batch_size=256):
        if not os.path.exists(anchor_file_path):
            raise FileNotFoundError(f"Anchor file not found: {anchor_file_path}")

        with open(anchor_file_path, "r") as f:
            anchors_dict = json.load(f)

        self.anchors_list = list(anchors_dict.values())

        class ModelArgs:
            def __init__(self, model_name_or_path):
                self.model_name_or_path = model_name_or_path

        model_args = ModelArgs(backbone_model)
        self.model = cos_simi_model(model_args, anchors=self.anchors_list, batch_size=batch_size, is_binary=False)

    def encode(self, sentences, **kwargs):
        return self.model.encode(sentences)


def get_model_wrapper(model_name):
    if model_name == "LDIR-UAE-500":
        anchor_file = checkpoints_dir / "LDIR-UAE-500" / "anchors.json"
        print(f"[INFO] Loading LDIR-UAE-500 from {anchor_file}")
        return LDIRModelWrapper(
            anchor_file_path=str(anchor_file),
            backbone_model="WhereIsAI/UAE-Large-V1",
            batch_size=256,
        )

    model_config = {
        "CQG-MBQA": {
            "path": "CQG-MBQA",
            "model_file": "mbqa_model.pt",
            "backbone": "WhereIsAI/UAE-Large-V1",
        },
        "QAEmb-MBQA": {
            "path": "QAEmb-MBQA",
            "model_file": "qaemb_model.pt",
            "backbone": "WhereIsAI/UAE-Large-V1",
        },
        "QIME": {
            "path": "QIME",
            "model_file": "qime_model_base.pt",
            "backbone": "abhinand/MedEmbed-large-v0.1",
        },
    }

    if model_name in model_config:
        config = model_config[model_name]
        base_path = checkpoints_dir / config["path"]
        questions_file = base_path / "questions.json"
        model_file = base_path / config["model_file"]

        print(f"[INFO] Loading {model_name}...")
        print(f"  - Questions: {questions_file}")
        print(f"  - Model: {model_file}")
        print(f"  - Backbone: {config['backbone']}")

        if not questions_file.exists():
            raise FileNotFoundError(f"Questions file not found: {questions_file}")
        if not model_file.exists():
            raise FileNotFoundError(f"Model file not found: {model_file}")

        with open(questions_file, "r") as f:
            linear_questions = json.load(f)

        model = IMECClassifier(num_labels=len(linear_questions), backbone=config["backbone"])
        map_location = "cuda:0" if device == "cuda" else "cpu"
        model.load_state_dict(torch.load(model_file, map_location=map_location))
        model.to(device)
        model.eval()

        return IMECMTEBModelWrapper(
            model,
            linear_questions,
            is_binary=True,
            is_sparse=False,
            binary_threshold=0.5,
            use_sigmoid=True,
        )

    if model_name == "bag_of_words":
        print("[INFO] Loading BagOfTokenEncoder")
        return BagOfTokenEncoder()

    print(f"[INFO] Loading general SBERT model: {model_name}")
    return SBERTEncodingModelGeneral(model_name, device=device)


def get_medical_tasks():
    return [
        mteb.get_task("BIOSSES", languages=["eng"]),
        mteb.get_task("R2MEDIIYiClinicalRetrieval", languages=["eng"]),
        mteb.get_task("R2MEDPMCClinicalRetrieval", languages=["eng"]),
        mteb.get_task("NFCorpus", languages=["eng"]),
        mteb.get_task("PublicHealthQA", languages=["eng"]),
        mteb.get_task("MedicalQARetrieval", languages=["eng"]),
        mteb.get_task("TRECCOVID", languages=["eng"]),
        mteb.get_task("BiorxivClusteringP2P", languages=["eng"]),
        mteb.get_task("BiorxivClusteringS2S", languages=["eng"]),
        mteb.get_task("MedrxivClusteringP2P", languages=["eng"]),
        mteb.get_task("MedrxivClusteringS2S", languages=["eng"]),
        mteb.get_task("ClusTREC-Covid", languages=["eng"]),
    ]


def evaluate_single_model(model_name, tasks):
    print(f"\n[INFO] Evaluating {model_name}")
    try:
        model_wrapper = get_model_wrapper(model_name)
        output_folder = model_outputs_dir / model_name.replace("/", "_")
        output_folder.mkdir(parents=True, exist_ok=True)

        completed_tasks = []
        for task in tasks:
            task_name = task.metadata.name
            print(f"  [INFO] Evaluating task: {task_name}")

            if hasattr(model_wrapper, "set_current_task"):
                model_wrapper.set_current_task(task_name)

            single_task_evaluation = mteb.MTEB(tasks=[task])
            single_task_evaluation.run(model_wrapper, output_folder=str(output_folder))
            completed_tasks.append(task_name)

        print(f"  [INFO] Completed: {len(completed_tasks)}/{len(tasks)} tasks")

        if hasattr(model_wrapper, "model") and hasattr(model_wrapper.model, "to"):
            model_wrapper.model.to("cpu")
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

        return completed_tasks
    except Exception as e:
        print(f"  [ERROR] {e}")
        import traceback
        traceback.print_exc()
        return None


def check_evaluation_completeness(models, tasks):
    missing_evaluations = []
    for model_name in models:
        model_path = model_outputs_dir / model_name.replace("/", "_") / "no_model_name_available" / "no_revision_available"
        if not model_path.exists():
            missing_evaluations.append(f"{model_name}: Directory not found")
            continue

        for task in tasks:
            task_file = model_path / f"{task.metadata.name}.json"
            if not task_file.exists():
                missing_evaluations.append(f"{model_name}: Missing {task.metadata.name}")

    return missing_evaluations


def generate_results_csv(models, tasks, metric_name="ndcg_at_10", output_suffix=""):
    report_dir.mkdir(parents=True, exist_ok=True)
    results_data = []

    for model_name in models:
        model_path = model_outputs_dir / model_name.replace("/", "_") / "no_model_name_available" / "no_revision_available"
        row = {"Model": model_name}
        task_scores = []

        for task in tasks:
            task_name = task.metadata.name
            task_type = task.metadata.type

            if metric_name == "main_metric":
                if task_name == "BIOSSES" or task_type == "STS":
                    target_metric = "spearman"
                elif "Clustering" in task_name or task_type == "Clustering":
                    target_metric = "v_measure"
                else:
                    target_metric = "ndcg_at_10"
            else:
                target_metric = metric_name

            task_file = model_path / f"{task_name}.json"
            custom_score = "N/A"

            if task_file.exists():
                try:
                    with open(task_file, "r") as f:
                        task_results = json.load(f)

                    if "scores" in task_results:
                        scores = task_results["scores"]
                        for score_key in ["test", "dev", "all"]:
                            if score_key not in scores:
                                continue

                            target_scores = scores[score_key]
                            if isinstance(target_scores, list) and len(target_scores) > 0:
                                target_scores = target_scores[0]

                            if isinstance(target_scores, dict):
                                if target_metric == "spearman" and "cos_sim" in target_scores and "spearman" in target_scores["cos_sim"]:
                                    custom_score = target_scores["cos_sim"]["spearman"]
                                    break

                                if target_metric in target_scores and target_scores[target_metric] is not None:
                                    custom_score = target_scores[target_metric]
                                    break
                except Exception as e:
                    print(f"[WARN] Error reading {task_file}: {e}")
                    custom_score = "Error"
            else:
                custom_score = "Missing"

            row[task_name] = custom_score
            if isinstance(custom_score, (int, float, np.floating)):
                task_scores.append(float(custom_score))

        row["Average"] = round(sum(task_scores) / len(task_scores), 4) if task_scores else "N/A"
        results_data.append(row)

    df = pd.DataFrame(results_data)
    task_columns = [task.metadata.name for task in tasks]
    df = df[[col for col in ["Model", *task_columns, "Average"] if col in df.columns]]

    if metric_name == "main_metric":
        csv_filename = "medical_mteb_main_results.csv"
    elif output_suffix:
        csv_filename = f"retrieval_{output_suffix}.csv"
    else:
        csv_filename = f"results_{metric_name}.csv"

    csv_path = report_dir / csv_filename
    df.to_csv(csv_path, index=False)
    print(f"Saved: {csv_path}")
    return df


def generate_all_metrics(models, tasks):
    generate_results_csv(models, tasks, metric_name="main_metric")
    for metric_name, suffix in [
        ("ndcg_at_10", "ndcg_at_10"),
        ("mrr_at_10", "mrr_at_10"),
        ("recall_at_10", "recall_at_10"),
    ]:
        generate_results_csv(models, tasks, metric_name=metric_name, output_suffix=suffix)

## Configuration

In [ ]:
SELECTED_MODELS = [
    "CQG-MBQA",
    "QAEmb-MBQA",
    "QIME",
    "LDIR-UAE-500",
    "bag_of_words",
    "google-bert/bert-base-uncased",
]

FORCE_RERUN = False
tasks = get_medical_tasks()

print(f"Selected models ({len(SELECTED_MODELS)}):")
for model_name in SELECTED_MODELS:
    print(f"- {model_name}")

print(f"\nSelected tasks ({len(tasks)}):")
for task in tasks:
    print(f"- {task.metadata.name} [{task.metadata.type}]")

## Local File Check

In [ ]:
required_files = {
    "CQG-MBQA": [
        checkpoints_dir / "CQG-MBQA" / "questions.json",
        checkpoints_dir / "CQG-MBQA" / "mbqa_model.pt",
    ],
    "QAEmb-MBQA": [
        checkpoints_dir / "QAEmb-MBQA" / "questions.json",
        checkpoints_dir / "QAEmb-MBQA" / "qaemb_model.pt",
    ],
    "QIME": [
        checkpoints_dir / "QIME" / "questions.json",
        checkpoints_dir / "QIME" / "qime_model_base.pt",
    ],
    "LDIR-UAE-500": [
        checkpoints_dir / "LDIR-UAE-500" / "anchors.json",
    ],
}

status_rows = []
for model_name, paths in required_files.items():
    for path in paths:
        status_rows.append({
            "Model": model_name,
            "Path": str(path),
            "Exists": path.exists(),
        })

pd.DataFrame(status_rows)

## Run Evaluation

In [ ]:
missing_evaluations = check_evaluation_completeness(SELECTED_MODELS, tasks)

if missing_evaluations:
    print("Missing evaluations:")
    for item in missing_evaluations:
        print(f"- {item}")
else:
    print("All selected evaluations are already complete.")

for model_name in SELECTED_MODELS:
    model_missing = [item for item in missing_evaluations if item.startswith(f"{model_name}:")]
    if FORCE_RERUN or model_missing:
        evaluate_single_model(model_name, tasks)
    else:
        print(f"Skipping {model_name}; results already exist.")

## Generate Reports

In [ ]:
main_results = generate_results_csv(SELECTED_MODELS, tasks, metric_name="main_metric")
generate_all_metrics(SELECTED_MODELS, tasks)
main_results

## Output Locations

In [ ]:
print("Model outputs:", model_outputs_dir)
print("Reports:", report_dir)